# Bulan 3: Enrichment & Validasi
## Minggu 9-10 (Domain-Specific Labeling)

Di tahap ini kita bakal ngasih konteks/label ke data yang udah bersih dari Bulan 2. Kita pake teknik **Keyword-based Labeling** buat ngelompokin data ke kategori tertentu.

In [ ]:
import pandas as pd
import os

# Baca hasil output dari Bulan 2
# Prioritas: pipeline akhir > dataset cleaned
file_pipeline = '../Bulan 2/pipeline_akhir_bulan_2.xlsx'
file_cleaned = '../Bulan 2/dataset_cleaned.xlsx'

if os.path.exists(file_pipeline):
    df = pd.read_excel(file_pipeline)
    text_col = 'normalized_text' if 'normalized_text' in df.columns else 'cleaned_text'
    print(f'Pake data pipeline akhir Bulan 2 ({len(df)} baris)')
elif os.path.exists(file_cleaned):
    df = pd.read_excel(file_cleaned)
    text_col = 'cleaned_text'
    print(f'Pake data cleaned Bulan 2 ({len(df)} baris)')
else:
    print('Dataset Bulan 2 belom ada! Jalanin notebook Bulan 2 dulu ya.')

df.head(3)

### 1. Definisikan Keyword dan Kategori (Rules)
Kita susun *dictionary* berisi daftar kata kunci buat masing-masing kategori kelas.

In [ ]:
# Definisi keyword buat tiap kategori
labeling_rules = {
    'Insomnia': ['tidur', 'lelah', 'begadang', 'ngantuk', 'insomnia', 'capek', 'melek'],
    'Depresi': ['sedih', 'nangis', 'menangis', 'putus asa', 'hancur', 'depresi', 'nyerah'],
    'Cemas': ['cemas', 'takut', 'khawatir', 'gugup', 'overthinking', 'panik', 'gelisah'],
    'Stress': ['stres', 'stress', 'pusing', 'muak', 'gila', 'beban', 'berat']
}

def apply_keyword_labeling(text):
    if not isinstance(text, str):
        return 'Tidak Terdefinisi'
    
    text = text.lower()
    
    # Cek setiap kategori dan keyword-nya
    for label, keywords in labeling_rules.items():
        for keyword in keywords:
            if keyword in text:
                return label
            
    return 'Lainnya / Netral'

### 2. Mengaplikasikan Labeling ke Dataset

In [ ]:
# Terapkan fungsi ke kolom teks
df['Kategori_Mental'] = df[text_col].apply(apply_keyword_labeling)

# Hitung distribusi label
label_counts = df['Kategori_Mental'].value_counts()
print('Distribusi Kategori Label:')
print(label_counts)

### 3. Evaluasi (Demo Data yang Berhasil Dilabeli)
Menampilkan contoh data yang terdeteksi masuk ke kategori tertentu (bukan Netral).

In [ ]:
pd.set_option('display.max_colwidth', None)
df_labeled_only = df[df['Kategori_Mental'] != 'Lainnya / Netral']

if len(df_labeled_only) > 0:
    demo_df = df_labeled_only[[text_col, 'Kategori_Mental']].head(10)
    display(demo_df)
else:
    print('Tidak ada data yang mengandung keyword di dataset ini.')

In [ ]:
# Simpan dataset yang sudah dilabeli
output_file = 'dataset_labeled_bulan3.xlsx'
df.to_excel(output_file, index=False)
print(f'Proses Labeling Selesai! {len(df)} baris data disimpan di: {output_file}')